<center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/DLI_Header_White.png" width="400" height="186" /></a></center>

<br>

# <font color="#76b900">**Notebook 7:** Retrieval-Augmented Generation with Vector Stores</font>

<br>

In the previous notebook, we learned about embedding models and exercised some of their capabilities. We discussed their intended use cases of longer-form document comparison and found ways to use it as a backbone for more custom semantic comparisons. This notebook will progress these ideas toward the retrieval model's intended use case and explore how to build chatbot systems that rely on *vector stores* to automatically save and retrieve information.

<br>

### **Learning Objectives:**

- Understand how semantic-similarity-backed systems can facilitate easy-to-use retrieval formulations.

- Learn how to incorporate retrieval modules into your chat model systems for a retrieval-augmented generation (RAG) pipeline, which can be applied to tasks like document retrieval and conversation memory buffers.

<br>

### **Questions To Think About:**

- This notebook does not attempt to incorporate hierarchical reasoning or non-naive RAG (such as planning agents). Consider what modifications would be necessary to make these components work in an LCEL chain.

- Consider when it would be best to move your vector store solution into a scalable service and when a GPU will become necessary for optimization.

<br>

### **Environment Setup:**

In [1]:
# %%capture
## ^^ Comment out if you want to see the pip install process

## Necessary for Colab, not necessary for course environment
# %pip install -q langchain langchain-nvidia-ai-endpoints gradio rich
# %pip install -q arxiv pymupdf faiss-cpu

## If you encounter a typing-extensions issue, restart your runtime and try again
# from langchain_nvidia_ai_endpoints import ChatNVIDIA
# ChatNVIDIA.get_available_models()

from functools import partial
from rich.console import Console
from rich.style import Style
from rich.theme import Theme

console = Console()
base_style = Style(color="#76B900", bold=True)
pprint = partial(console.print, style=base_style)

In [2]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings

# NVIDIAEmbeddings.get_available_models()
embedder = NVIDIAEmbeddings(model="nvidia/nv-embed-v1", truncate="END")

# ChatNVIDIA.get_available_models()
instruct_llm = ChatNVIDIA(model="mistralai/mixtral-8x22b-instruct-v0.1")

----

<br>

## Part 1: Summary of RAG Workflows

This notebook will explore several paradigms and derive reference code to help you approach some of the most common retrieval-augmented workflows. Specifically, the following sections will be covered (with the differences highlighted):

<br>

> ***Vector Store Workflow for Conversational Exchanges:***
- Generate semantic embedding for each new conversation.
- Add the message body to a vector store for retrieval.
- Query the vector store for relevant messages to fill in the LLM context.

<br>

> ***Modified Workflow for an Arbitrary Document:***
- **Divide the document into chunks and process them into useful messages.**
- Generate semantic embedding for each **new document chunk**.
- Add the **chunk bodies** to a vector store for retrieval.
- Query the vector store for relevant **chunks** to fill in the LLM context.
    - ***Optional:* Modify/synthesize results for better LLM results.**

<br>

> **Extended Workflow for a Directory of Arbitrary Documents:**
- Divide **each document** into chunks and process them into useful messages.
- Generate semantic embedding for each new document chunk.
- Add the chunk bodies to **a scalable vector database for fast retrieval**.
    - ***Optional*: Exploit hierarchical or metadata structures for larger systems.**
- Query the **vector database** for relevant chunks to fill in the LLM context.
    - *Optional:* Modify/synthesize results for better LLM results.

<br>

Some of the most important terminology surrounding RAG is covered in detail on the [**LlamaIndex Concepts page**](https://docs.llamaindex.ai/en/stable/getting_started/concepts.html), which itself is a great starting point for progressing towards the LlamaIndex loading and retrieving strategy. We highly recommend using it as a reference as you continue with this notebook and advise you to try out LlamaIndex after the course to consider the pros and cons firsthand!


<!-- > <img src="https://drive.google.com/uc?export=view&id=1cFbKbVvLLnFPs3yWCKIuzXkhBWh6nLQY" width=1200px/> -->
> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/data_connection_langchain.jpeg" width=1200px/>
>
> From [**Retrieval | LangChain**🦜️🔗](https://python.langchain.com/v0.1/docs/modules/data_connection/)

----

<br>

## **Part 2:** RAG for Conversation History

In our previous explorations, we delved into the capabilities of document embedding models and used them to embed, store, and compare semantic vector representations of text. Though we could motivate how to efficiently extend this into vector store land manually, the true beauty of working with a standard API is its strong incorporation with other frameworks that can already do the heavy lifting for us!

<br>

### **Step 1**: Getting A Conversation

Consider a conversation crafted using Llama-13B between a chat agent and a blue bear named Beras. This dialogue, dense with details and potential diversions, provides a rich dataset for our study:


In [3]:
conversation = [  ## This conversation was generated partially by an AI system, and modified to exhibit desirable properties
    "[User]  Hello! My name is Beras, and I'm a big blue bear! Can you please tell me about the rocky mountains?",
    "[Agent] The Rocky Mountains are a beautiful and majestic range of mountains that stretch across North America",
    "[Beras] Wow, that sounds amazing! Ive never been to the Rocky Mountains before, but Ive heard many great things about them.",
    "[Agent] I hope you get to visit them someday, Beras! It would be a great adventure for you!"
    "[Beras] Thank you for the suggestion! Ill definitely keep it in mind for the future.",
    "[Agent] In the meantime, you can learn more about the Rocky Mountains by doing some research online or watching documentaries about them."
    "[Beras] I live in the arctic, so I'm not used to the warm climate there. I was just curious, ya know!",
    "[Agent] Absolutely! Lets continue the conversation and explore more about the Rocky Mountains and their significance!"
]

Using the manual embedding strategy from the previous notebook is still very viable, but we can also rest easy and let a **vector store** do all that work for us!

<br>

### **Step 2:** Constructing Our Vector Store Retriever

To streamline similarity queries on our conversation, we can employ a vector store to help keep track of passages for us! **Vector Stores**, or vector storage systems, abstract away most of the low-level details of the embedding/comparison strategies and provide a simple interface to load and compare vectors.


<!-- > <img src="https://drive.google.com/uc?export=view&id=1ZjwYbSZzsXK6ZP8O1-cY3BeRffV4oqzb" width=1000px/> -->
> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/vector_stores.jpeg" width=1200px/>
>
> From [**Vector Stores | LangChain**🦜️🔗](https://python.langchain.com/v0.1/docs/modules/data_connection/vectorstores/)

<br>

In addition to simplifying the process from an API perspective, vector stores also implement connectors, integrations, and optimizations under the hood. In our case, we will start with the [**FAISS vector store**](https://python.langchain.com/docs/integrations/vectorstores/faiss), which integrates a LangChain-compatable Embedding model with the [**FAISS (Facebook AI Similarity Search)**](https://github.com/facebookresearch/faiss) library to make the process fast and scalable on our local machine!

**Specifically:**

1. We can feed our conversation into [**a FAISS vector store**](https://python.langchain.com/docs/integrations/vectorstores/faiss) via the `from_texts` constructor. This will take our conversational data and the embedding model to create a searchable index over our discussion.
2. This vector store can then be "interpreted" as a retriever, supporting the LangChain runnable API and returning documents retrieved via an input query.

The following shows how you can construct a FAISS vector store and reinterpret it as a retriever using the LangChain `vectorstore` API:

In [4]:
%%time
## ^^ This cell will be timed to see how long the conversation embedding takes
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain.vectorstores import FAISS

## Streamlined from_texts FAISS vectorstore construction from text list
convstore = FAISS.from_texts(conversation, embedding=embedder)
retriever = convstore.as_retriever()

CPU times: user 198 ms, sys: 351 ms, total: 550 ms
Wall time: 3.2 s


The retriever can now be used like any other LangChain runnable to query the vector store for some relevant documents:

In [5]:
pprint(retriever.invoke("What is your name?"))

[
    Document(
        id='ff7464d8-a7e1-44ce-8fbc-7c1404c60ac1',
        metadata={},
        page_content="[User]  Hello! My name is Beras, and I'm a big blue bear! Can you please tell me about the 
rocky mountains?"
    ),
    Document(
        id='65ffb13f-e4b0-4964-89c6-95df15c43f2d',
        metadata={},
        page_content='[Agent] Absolutely! Lets continue the conversation and explore more about the Rocky Mountains
and their significance!'
    ),
    Document(
        id='50f874fb-1f19-4ae1-9730-782633d7e76a',
        metadata={},
        page_content='[Agent] I hope you get to visit them someday, Beras! It would be a great adventure for 
you![Beras] Thank you for the suggestion! Ill definitely keep it in mind for the future.'
    ),
    Document(
        id='c742a606-86db-4290-8530-1259b870458f',
        metadata={},
        page_content="[Agent] In the meantime, you can learn more about the Rocky Mountains by doing some research 
online or watching documentaries about them.[Beras] I live in the arctic, so I'm not used to the warm climate 
there. I was just curious, ya know!"
    )
]

In [6]:
pprint(retriever.invoke("Where are the Rocky Mountains?"))

[
    Document(
        id='d0fe86d7-516c-45a6-bdd0-edc6a6f85112',
        metadata={},
        page_content='[Agent] The Rocky Mountains are a beautiful and majestic range of mountains that stretch 
across North America'
    ),
    Document(
        id='c742a606-86db-4290-8530-1259b870458f',
        metadata={},
        page_content="[Agent] In the meantime, you can learn more about the Rocky Mountains by doing some research 
online or watching documentaries about them.[Beras] I live in the arctic, so I'm not used to the warm climate 
there. I was just curious, ya know!"
    ),
    Document(
        id='ff7464d8-a7e1-44ce-8fbc-7c1404c60ac1',
        metadata={},
        page_content="[User]  Hello! My name is Beras, and I'm a big blue bear! Can you please tell me about the 
rocky mountains?"
    ),
    Document(
        id='65ffb13f-e4b0-4964-89c6-95df15c43f2d',
        metadata={},
        page_content='[Agent] Absolutely! Lets continue the conversation and explore more about the Rocky Mountains
and their significance!'
    )
]

As we can see, our retriever found a handful of semantically relevant documents from our query. You may notice that not all of the documents are useful or clear on their own. For example, a retrieval of *"Beras"* for *"your name"* may be problematic for the chatbot if provided out of context. Anticipating the potential problems and creating synergies between your LLM components can increase the likelihood of good RAG behavior, so keep an eye out for such pitfalls and opportunities.

<br>

### **Step 3:** Incorporating Conversation Retrieval Into Our Chain

Now that we have our loaded retriever component as a chain, we can incorporate it into our existing chat system as before. Specifically, we can start with an ***always-on RAG formulation*** where:
- **A retriever is always retrieving context by default**.
- **A generator is acting on the retrieved context**.

In [7]:
from langchain.document_transformers import LongContextReorder
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnableLambda
from langchain.schema.runnable.passthrough import RunnableAssign
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings

from functools import partial
from operator import itemgetter

########################################################################
## Utility Runnables/Methods
def RPrint(preface=""):
    """Simple passthrough "prints, then returns" chain"""
    def print_and_return(x, preface):
        if preface: print(preface, end="")
        pprint(x)
        return x
    return RunnableLambda(partial(print_and_return, preface=preface))

def docs2str(docs, title="Document"):
    """Useful utility for making chunks into context string. Optional, but useful"""
    out_str = ""
    for doc in docs:
        doc_name = getattr(doc, 'metadata', {}).get('Title', title)
        if doc_name:
            out_str += f"[Quote from {doc_name}] "
        out_str += getattr(doc, 'page_content', str(doc)) + "\n"
    return out_str

## Optional; Reorders longer documents to center of output text
long_reorder = RunnableLambda(LongContextReorder().transform_documents)

In [8]:
context_prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context"
    "\n\nRetrieved Context: {context}"
    "\n\nUser Question: {question}"
    "\nAnswer the user conversationally. User is not aware of context."
)

chain = (
    {
        'context': convstore.as_retriever() | long_reorder | docs2str,
        'question': (lambda x:x)
    }
    | context_prompt
    # | RPrint()
    | instruct_llm
    | StrOutputParser()
)

pprint(chain.invoke("Where does Beras live?"))

Based on the context provided, Beras lives in the Arctic. It seems like they're not very familiar with the warm 
climate of the Rocky Mountains.

Take a second to try out some more invocations and see how the new setup performs. Regardless of your model choice, the following questions should serve as interesting starting points.

In [9]:
pprint(chain.invoke("Where are the Rocky Mountains?"))

Hello there! The Rocky Mountains are a stunning range of mountains that span across North America. Unfortunately, 
without more context, I can't give you a precise location beyond that. However, if you're interested, you can learn
more about them by doing some online research or watching informative documentaries!

In [10]:
pprint(chain.invoke("Where are the Rocky Mountains? Are they close to California?"))

The Rocky Mountains are a stunning mountain range that extends across North America. While they're not directly 
close to California, they do reach as far south as New Mexico. If you're traveling from California, you could 
certainly visit some parts of the Rocky Mountains with a bit of journeying. They're truly a sight to behold!

In [11]:
pprint(chain.invoke("How far away is Beras from the Rocky Mountains?"))

Unfortunately, the context doesn't provide information on Beras' exact location or distance from the Rocky 
Mountains. I do know that Beras mentioned living in the Arctic, which is quite far from the Rockies geographically.
The Rocky Mountains are a vast mountain range located mostly in the western United States and Canada.

<br>

You might notice some decent performance with this always-on retrieval node in the loop since the actual context being fed into the LLM remains relatively small. It's important to experiment with factors like embedding sizes, context limits, and model options to see what kinds of behavior you can expect and which efforts are worth taking to improve performance.

<br>

### **Step 4:** Automatic Conversation Storage

Now that we see how our vector store memory unit should function, we can perform one last integration to allow our conversation to add new entries to our conversation: a runnable that calls the `add_texts` method for us to update the store state.


In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter

########################################################################
## Reset knowledge base and define what it means to add more messages.
convstore = FAISS.from_texts(conversation, embedding=embedder)

def save_memory_and_get_output(d, vstore):
    """Accepts 'input'/'output' dictionary and saves to convstore"""
    vstore.add_texts([f"User said {d.get('input')}", f"Agent said {d.get('output')}"])
    return d.get('output')

########################################################################

# instruct_llm = ChatNVIDIA(model="mistralai/mixtral-8x22b-instruct-v0.1")

chat_prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context"
    "\n\nRetrieved Context: {context}"
    "\n\nUser Question: {input}"
    "\nAnswer the user conversationally. Make sure the conversation flows naturally.\n"
    "[Agent]"
)


conv_chain = (
    {
        'context': convstore.as_retriever() | long_reorder | docs2str,
        'input': (lambda x:x)
    }
    | RunnableAssign({'output' : chat_prompt | instruct_llm | StrOutputParser()})
    | partial(save_memory_and_get_output, vstore=convstore)
)

pprint(conv_chain.invoke("I'm glad you agree! I can't wait to get some ice cream there! It's such a good food!"))
print()
pprint(conv_chain.invoke("Can you guess what my favorite food is?"))
print()
pprint(conv_chain.invoke("Actually, my favorite is honey! Not sure where you got that idea?"))
print()
pprint(conv_chain.invoke("I see! Fair enough! Do you know my favorite food now?"))

While I'm sure the Rocky Mountains would provide a magnificent backdrop for enjoying ice cream, it seems like your 
excitement has taken a tasty turn! The Rockies are indeed awe-inspiring, spanning North America and offering 
breathtaking views and adventures. However, let's not forget about that delicious ice cream! While it may not be a 
traditional part of the Rockies experience, there's no reason we can't incorporate it into our daydreams about 
visiting this stunning mountain range! It's always great to keep our taste buds in mind while planning our 
explorations.

Given our conversation, I'd wager a guess that ice cream might just be your favorite food! After all, you did 
express your enthusiasm about enjoying some ice cream amidst the stunning views of the Rocky Mountains. Remember, 
though, this is just a friendly guess based on our recent chat.

I see! Well, I must admit, the joy you shared about savoring ice cream while surrounded by the awe-inspiring views 
of the Rocky Mountains led me to believe it was your favorite. Ice cream is indeed delightful, but honey is just as
wonderful! Just like how the Rockies can't be fully appreciated without seeing them firsthand, the true essence of 
honey is best experienced by tasting it. It's always exciting to discover more about one's preferences, isn't it? 
The world is full of sweet surprises, and honey is definitely one of them.

Of course! Based on our conversation, I now know that honey is your favorite food! It turns out, my guess about ice
cream wasn't quite on the mark, but it led to a sweet discovery nonetheless. Now, I'm even more curious to hear 
about other foods you cherish. Is there any specific reason why honey holds a special place in your heart?

Unlike the more automatic full-text or rule-based approaches to injecting context into the LLM, this approach ensures some amount of consolidation which can keep the context length from getting out of hand. It's still not a full-proof strategy on its own, but it's a stark improvement for unstructured conversations (and doesn't even require a strong instruction-tuned model to perform slot-filling).

----

<br>

## **Part 3 [Exercise]:** RAG For Document Chunk Retrieval

Given our prior exploration of document loading, the idea that data chunks can be embedded and searched through probably isn't surprising. With that said, it is definitely worth going over since applying RAG with documents is a double-edged sword; it may **seem** to work well out of the box but requires some extra care when optimizing it for truly reliable performance. It also provides an excellent opportunity to review some fundamental LCEL skills, so let's see what we can do!

<br>

### **Exercise:**

In the previous example, you may recall that we pulled in some relatively small papers with the help of [`ArxivLoader`](https://python.langchain.com/docs/integrations/document_loaders/arxiv) using the following syntax:

```python
from langchain.document_loaders import ArxivLoader

docs = [
    ArxivLoader(query="2205.00445").load(),  ## MRKL
    ArxivLoader(query="2210.03629").load(),  ## ReAct
]
```

Given all that you've learned so far, choose a selection of papers that you would like to use and develop a chatbot that can talk about them!

<br>

Though this is a pretty big task, a walkthrough of ***most*** of the process will be provided below. By the end of the walkthrough, many of the necessary puzzle pieces will be provided, and your real task will be to integrate them together for the final `retrieval_chain`. When you're done, get ready to re-integrate the chain (or a flavor of your choice) in the last notebook as part of the evaluation exercise!


<br>

### **Task 1**: Loading And Chunking Your Documents

The following code block gives you some default papers to load in for your RAG chain. Feel free to select more papers as desired, but note that longer documents will take longer to process. A few simplifying assumptions and additional processing steps are included to help you improve your naive RAG performance:

- Documents are cut off prior to the "References" section if one exists. This will keep our system from considering the citations and appendix sections, which tend to be long and distracting.

- A chunk that lists the available documents is inserted to provide a high-level view of all available documents in a single chunk. If your pipeline does not provide metadata on each retrieval, this is a useful component and can even be listed among a list of higher-priority pieces if appropriate.

- Additionally, the metadata entries are also inserted to provide general information. Ideally, there would also be some synthetic chunks that merge the metadata into interesting cross-document chunks.

**NOTE:** ***For the sake of the assessment, please include at least one paper that is less than one month old!***


In [13]:
import json
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings

from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import ArxivLoader

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=100,
    separators=["\n\n", "\n", ".", ";", ",", " "],
)

## TODO: Please pick some papers and add them to the list as you'd like
## NOTE: To re-use for the final assessment, make sure at least one paper is < 1 month old
print("Loading Documents")
docs = [
    ArxivLoader(query="2005.11401").load(),  ## RAG Paper   
    ArxivLoader(query="2510.10448").load()
    # ArxivLoader(query="1706.03762").load(),  ## Attention Is All You Need Paper
    # ArxivLoader(query="1810.04805").load(),  ## BERT Paper
    # ArxivLoader(query="2005.11401").load(),  ## RAG Paper
    # ArxivLoader(query="2205.00445").load(),  ## MRKL Paper
    # ArxivLoader(query="2310.06825").load(),  ## Mistral Paper
    # ArxivLoader(query="2306.05685").load(),  ## LLM-as-a-Judge
    ## Some longer papers
    # ArxivLoader(query="2210.03629").load(),  ## ReAct Paper
    # ArxivLoader(query="2112.10752").load(),  ## Latent Stable Diffusion Paper
    # ArxivLoader(query="2103.00020").load(),  ## CLIP Paper
    ## TODO: Feel free to add more
]

## Cut the paper short if references is included.
## This is a standard string in papers.
for doc in docs:
    content = json.dumps(doc[0].page_content)
    if "References" in content:
        doc[0].page_content = content[:content.index("References")]

## Split the documents and also filter out stubs (overly short chunks)
print("Chunking Documents")
docs_chunks = [text_splitter.split_documents(doc) for doc in docs]
docs_chunks = [[c for c in dchunks if len(c.page_content) > 200] for dchunks in docs_chunks]

## Make some custom Chunks to give big-picture details
doc_string = "Available Documents:"
doc_metadata = []
for chunks in docs_chunks:
    metadata = getattr(chunks[0], 'metadata', {})
    doc_string += "\n - " + metadata.get('Title')
    doc_metadata += [str(metadata)]

extra_chunks = [doc_string] + doc_metadata

## Printing out some summary information for reference
pprint(doc_string, '\n')
for i, chunks in enumerate(docs_chunks):
    print(f"Document {i}")
    print(f" - # Chunks: {len(chunks)}")
    print(f" - Metadata: ")
    pprint(chunks[0].metadata)
    print()

Loading Documents
Chunking Documents


Available Documents:
 - Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks
 - RECON: Reasoning with Condensation for Efficient Retrieval-Augmented Generation 

Document 0
 - # Chunks: 46
 - Metadata: 


{
    'Published': '2021-04-12',
    'Title': 'Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks',
    'Authors': 'Patrick Lewis, Ethan Perez, Aleksandra Piktus, Fabio Petroni, Vladimir Karpukhin, Naman Goyal, 
Heinrich Küttler, Mike Lewis, Wen-tau Yih, Tim Rocktäschel, Sebastian Riedel, Douwe Kiela',
    'Summary': 'Large pre-trained language models have been shown to store factual knowledge\nin their parameters, 
and achieve state-of-the-art results when fine-tuned on\ndownstream NLP tasks. However, their ability to access and
precisely manipulate\nknowledge is still limited, and hence on knowledge-intensive tasks, their\nperformance lags 
behind task-specific architectures. Additionally, providing\nprovenance for their decisions and updating their 
world knowledge remain open\nresearch problems. Pre-trained models with a differentiable access mechanism 
to\nexplicit non-parametric memory can overcome this issue, but have so far been\nonly investigated for extractive 
downstream tasks. We explore a general-purpose\nfine-tuning recipe for retrieval-augmented generation (RAG) -- 
models which\ncombine pre-trained parametric and non-parametric memory for language\ngeneration. We introduce RAG 
models where the parametric memory is a\npre-trained seq2seq model and the non-parametric memory is a dense vector 
index\nof Wikipedia, accessed with a pre-trained neural retriever. We compare two RAG\nformulations, one which 
conditions on the same retrieved passages across the\nwhole generated sequence, the other can use different 
passages per token. We\nfine-tune and evaluate our models on a wide range of knowledge-intensive NLP\ntasks and set
the state-of-the-art on three open domain QA tasks, outperforming\nparametric seq2seq models and task-specific 
retrieve-and-extract architectures.\nFor language generation tasks, we find that RAG models generate more 
specific,\ndiverse and factual language than a state-of-the-art parametric-only seq2seq\nbaseline.'
}


Document 1
 - # Chunks: 25
 - Metadata: 


{
    'Published': '2025-10-12',
    'Title': 'RECON: Reasoning with Condensation for Efficient Retrieval-Augmented Generation',
    'Authors': 'Zhichao Xu, Minheng Wang, Yawei Wang, Wenqian Ye, Yuntao Du, Yunpu Ma, Yijun Tian',
    'Summary': 'Retrieval-augmented generation (RAG) systems trained using reinforcement\nlearning (RL) with 
reasoning are hampered by inefficient context management,\nwhere long, noisy retrieved documents increase costs and
degrade performance.\nWe introduce RECON (REasoning with CONdensation), a framework that integrates\nan explicit 
summarization module to compress evidence within the reasoning\nloop. Our summarizer is trained via a two-stage 
process: relevance pretraining\non QA datasets, followed by multi-aspect distillation from proprietary LLMs 
to\nensure factuality and clarity. Integrated into the Search-R1 pipeline, RECON\nreduces total context length by 
35\\%, leading to improved training speed and\ninference latency, while simultaneously improving RAG performance on
downstream\nQA benchmarks. Notably, it boosts the average EM score of the 3B model by\n14.5\\% and the 7B model by 
3.0\\%, showing particular strength in multi-hop QA.\nRECON demonstrates that learned context compression is 
essential for building\npractical, scalable, and performant RAG systems. Our code implementation is\nmade available
at https://github.com/allfornancy/RECON.'
}

<br>

### **Task 2**: Construct Your Document Vector Stores

Now that we have all of the components, we can go ahead and create indices surrounding them:

In [14]:
%%time
print("Constructing Vector Stores")
vecstores = [FAISS.from_texts(extra_chunks, embedder)]
vecstores += [FAISS.from_documents(doc_chunks, embedder) for doc_chunks in docs_chunks]

Constructing Vector Stores
CPU times: user 173 ms, sys: 12.7 ms, total: 185 ms
Wall time: 7.27 s


<br>

From there, we can combine our indices into a single one using the following utility:

In [15]:
from faiss import IndexFlatL2
from langchain_community.docstore.in_memory import InMemoryDocstore

embed_dims = len(embedder.embed_query("test"))
def default_FAISS():
    '''Useful utility for making an empty FAISS vectorstore'''
    return FAISS(
        embedding_function=embedder,
        index=IndexFlatL2(embed_dims),
        docstore=InMemoryDocstore(),
        index_to_docstore_id={},
        normalize_L2=False
    )

def aggregate_vstores(vectorstores):
    ## Initialize an empty FAISS Index and merge others into it
    ## We'll use default_faiss for simplicity, though it's tied to your embedder by reference
    agg_vstore = default_FAISS()
    for vstore in vectorstores:
        agg_vstore.merge_from(vstore)
    return agg_vstore

## Unintuitive optimization; merge_from seems to optimize constituent vector stores away
docstore = aggregate_vstores(vecstores)

print(f"Constructed aggregate docstore with {len(docstore.docstore._dict)} chunks")

Constructed aggregate docstore with 74 chunks


<br>

### **Task 3: [Exercise]** Implement Your RAG Chain

Finally, all the puzzle pieces are in place to implement the RAG pipeline! As a review, we now have:

- A way to construct a from-scratch vector store for conversational memory (and a way to initialize an empty one with `default_FAISS()`)

- A vector store pre-loaded with useful document information from our `ArxivLoader` utility (stored in `docstore`).

With the help of a couple more utilities, you're finally ready to integrate your chain! A few additional convenience utilities are provided (`doc2str` and the now-common `RPrint`) but are optional to use. Additionally, some starter prompts and structures are also defined.

> **Given all of this:** Please implement the `retrieval_chain`.

In [16]:
from langchain.document_transformers import LongContextReorder
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.passthrough import RunnableAssign
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

import gradio as gr
from functools import partial
from operator import itemgetter

# NVIDIAEmbeddings.get_available_models()
embedder = NVIDIAEmbeddings(model="nvidia/nv-embed-v1", truncate="END")
# ChatNVIDIA.get_available_models()
instruct_llm = ChatNVIDIA(model="mistralai/mixtral-8x7b-instruct-v0.1")
# instruct_llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct")

convstore = default_FAISS()

def save_memory_and_get_output(d, vstore):
    """Accepts 'input'/'output' dictionary and saves to convstore"""
    vstore.add_texts([
        f"User previously responded with {d.get('input')}",
        f"Agent previously responded with {d.get('output')}"
    ])
    return d.get('output')

initial_msg = (
    "Hello! I am a document chat agent here to help the user!"
    f" I have access to the following documents: {doc_string}\n\nHow can I help you?"
)

chat_prompt = ChatPromptTemplate.from_messages([("system",
    "You are a document chatbot. Help the user as they ask questions about documents."
    " User messaged just asked: {input}\n\n"
    " From this, we have retrieved the following potentially-useful info: "
    " Conversation History Retrieval:\n{history}\n\n"
    " Document Retrieval:\n{context}\n\n"
    " (Answer only from retrieval. Only cite sources that are used. Make your response conversational.)"
), ('user', '{input}')])

stream_chain = chat_prompt| RPrint() | instruct_llm | StrOutputParser()

################################################################################################
## BEGIN TODO: Implement the retrieval chain to make your system work!

retrieval_chain = (
    {'input': (lambda x: x)}  # wrap user input
    | RunnableAssign({
        # Retrieve conversation history from convstore
        'history': lambda d: convstore.as_retriever().invoke(d['input']),
        # Retrieve relevant document context from docstore
        'context': lambda d: docs2str(
            LongContextReorder().transform_documents(
                docstore.as_retriever().invoke(d['input'])
            )
        )
    })
)

## END TODO
################################################################################################

def chat_gen(message, history=[], return_buffer=True):
    buffer = ""
    ## First perform the retrieval based on the input message
    retrieval = retrieval_chain.invoke(message)
    line_buffer = ""

    ## Then, stream the results of the stream_chain
    for token in stream_chain.stream(retrieval):
        buffer += token
        ## If you're using standard print, keep line from getting too long
        yield buffer if return_buffer else token

    ## Lastly, save the chat exchange to the conversation memory buffer
    save_memory_and_get_output({'input':  message, 'output': buffer}, convstore)


## Start of Agent Event Loop
test_question = "Tell me about RAG!"  ## <- modify as desired

## Before you launch your gradio interface, make sure your thing works
for response in chat_gen(test_question, return_buffer=False):
    print(response, end='')

ChatPromptValue(
    messages=[
        SystemMessage(
            content='You are a document chatbot. Help the user as they ask questions about documents. User messaged
just asked: Tell me about RAG!\n\n From this, we have retrieved the following potentially-useful info:  
Conversation History Retrieval:\n[]\n\n Document Retrieval:\n[Quote from RECON: Reasoning with Condensation for 
Efficient Retrieval-Augmented Generation] . IRCoT (Trivedi et al., 2023) interleaves\\nretrieval with steps in 
chain-of-thought prompting,\\nenabling more effective reasoning for knowledge-\\nintensive tasks that require 
multi-hop inference.\\nSelf-RAG (Asai et al., 2024) introduces super-\\nvised fine-tuning to train the generator 
LLM to au-\\ntonomously decide whether to retrieve or generate\\nat each step. Leveraging large-scale synthetic 
data,\\nSelf-RAG achieves state-of-the-art performance\\nacross various QA and long-form generation bench-\\nmarks 
(Gao et al., 2023a; Min et al., 2023).\\nMotivated by recent advances in reasoning-\\naware reinforcement learning 
(Guo et al., 2024;\\nHurst et al., 2024; Guo et al., 2025), several concur-\\nrent works explore training RAG 
agents via rein-\\nforcement learning (Chen et al., 2025; Song et al.,\\n2025; Zheng et al., 2025; Jin et al., 
2025c). Among\\nthese, we focus on Search-R1 (Jin et al., 2025c)\n[Quote from Retrieval-Augmented Generation for 
Knowledge-Intensive NLP Tasks] . We refer to this decoding procedure as \\u201cThorough Decoding.\\u201d For 
longer\\noutput sequences, |Y | can become large, requiring many forward passes. For more ef\\ufb01cient 
decoding,\\nwe can make a further approximation that p\\u03b8(y|x, zi) \\u22480 where y was not generated during 
beam\\nsearch from x, zi. This avoids the need to run additional forward passes once the candidate set Y has\\nbeen
generated. We refer to this decoding procedure as \\u201cFast Decoding.\\u201d\\n3\\nExperiments\\nWe experiment 
with RAG in a wide range of knowledge-intensive tasks. For all experiments, we use\\na single Wikipedia dump for 
our non-parametric knowledge source. Following Lee et al. [31] and\\nKarpukhin et al. [26], we use the December 
2018 dump. Each Wikipedia article is split into disjoint\\n100-word chunks, to make a total of 21M 
documents\n[Quote from RECON: Reasoning with Condensation for Efficient Retrieval-Augmented Generation] . Further, 
RECON achieves\\nfaster training speed (5.2%) and lower inference la-\\ntency (30.9%) measured in wall-clock time, 
demon-\\nstrating that active context compression is critical\\ntoward practical, RL-augmented RAG 
systems.\\n2\\nRelated Works\\nRAG and Agentic RAG.\\nRetrieval-Augmented\\nGeneration (RAG, Lewis et al., 2020b) 
enhances\\nlarge language models (LLMs) by retrieving rel-\\nevant document chunks from external knowledge\\nbases 
(Xu et al., 2025). RAG helps mitigate hal-\\nlucination by grounding generation in retrieved\\ncontent, and has 
become a foundational technique\\nin developing chatbots and real-world LLM ap-\\nplications (Gao et al., 2023b). 
Beyond the stan-\\ndard retrieve-then-generate pipeline (often referred\\nto as Naive RAG), numerous studies have 
pro-\\nposed techniques to improve performance and ef-\\nficiency. IRCoT (Trivedi et al\n[Quote from 
Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks] . In one approach, RAG-Sequence, the model uses 
the same document\\nto predict each target token. The second approach, RAG-Token, can predict each target token 
based\\non a different document. In the following, we formally introduce both models and then describe 
the\\np\\u03b7 and p\\u03b8 components, as well as the training and decoding 
procedure.\\n2.1\\nModels\\nRAG-Sequence Model\\nThe RAG-Sequence model uses the same retrieved document to 
generate\\nthe complete sequence. Technically, it treats the retrieved document as a single latent variable 
that\\nis marginalized to get the seq2seq probability p(y|x) via a to

RAG, or Retrieval-Augmented Generation, is a technique that enhances large language models by retrieving relevant document chunks from external knowledge bases. This helps to mitigate the issue of hallucination and has become a foundational technique in developing chatbots and real-world LLM applications. The retrieved content is grounded in the generation process, making the output more reliable and accurate.

In the original RAG model, there are two main steps: retrieval and generation. During the retrieval phase, a set of documents is gathered from a knowledge source based on the input prompt. Then, during the generation phase, the language model generates the output sequence using the retrieved documents as additional context.

There are also variations of the RAG model like RAG-Sequence and RAG-Token. The RAG-Sequence model uses the same retrieved document to generate the complete sequence, while the RAG-Token model can predict each target token based on a different document.

Rec

### **Task 4:** Interact With Your Gradio Chatbot

In [17]:
chatbot = gr.Chatbot(value = [[None, initial_msg]])
demo = gr.ChatInterface(chat_gen, chatbot=chatbot).queue()

try:
    demo.launch(debug=True, share=True, show_api=False)
    demo.close()
except Exception as e:
    demo.close()
    print(e)
    raise e

/usr/local/lib/python3.11/site-packages/gradio/analytics.py:106: UserWarning: IMPORTANT: You are using gradio version 4.41.0, however version 4.44.1 is available, please upgrade. 
--------
  warnings.warn(


Running on local URL:  http://127.0.0.1:7860
Running on public URL: https://80a97684fea93d9699.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


ChatPromptValue(
    messages=[
        SystemMessage(
            content="You are a document chatbot. Help the user as they ask questions about documents. User messaged
just asked: what is RAG\n\n From this, we have retrieved the following potentially-useful info:  Conversation 
History Retrieval:\n[Document(id='602a4fea-bd49-42b1-a334-6bbf6596c1d1', metadata={}, page_content='Agent 
previously responded with RAG, or Retrieval-Augmented Generation, is a technique that enhances large language 
models by retrieving relevant document chunks from external knowledge bases. This helps to mitigate the issue of 
hallucination and has become a foundational technique in developing chatbots and real-world LLM applications. The 
retrieved content is grounded in the generation process, making the output more reliable and accurate.\\n\\nIn the 
original RAG model, there are two main steps: retrieval and generation. During the retrieval phase, a set of 
documents is gathered from a knowledge source based on the input prompt. Then, during the generation phase, the 
language model generates the output sequence using the retrieved documents as additional context.\\n\\nThere are 
also variations of the RAG model like RAG-Sequence and RAG-Token. The RAG-Sequence model uses the same retrieved 
document to generate the complete sequence, while the RAG-Token model can predict each target token based on a 
different document.\\n\\nRecent improvements to RAG include IRCoT, which interleaves retrieval with steps in 
chain-of-thought prompting to enable more effective reasoning for knowledge-intensive tasks that require multi-hop 
inference. Self-RAG introduces supervised fine-tuning to train the generator language model to autonomously decide 
whether to retrieve or generate at each step, achieving state-of-the-art performance on various QA and long-form 
generation benchmarks.\\n\\n(Sources: Lewis et al., 2020b; Trivedi et al.; Gao et al., 2023a; Min et al., 2023; 
Chen et al., 2025; Song et al., 2025; Zheng et al., 2025; Jin et al., 2025c; Asai et al., 2024)'), 
Document(id='901f7976-7223-4112-b597-66728d6be6a6', metadata={}, page_content='User previously responded with Tell 
me about RAG!')]\n\n Document Retrieval:\n[Quote from RECON: Reasoning with Condensation for Efficient 
Retrieval-Augmented Generation] . Further, RECON achieves\\nfaster training speed (5.2%) and lower inference 
la-\\ntency (30.9%) measured in wall-clock time, demon-\\nstrating that active context compression is 
critical\\ntoward practical, RL-augmented RAG systems.\\n2\\nRelated Works\\nRAG and Agentic 
RAG.\\nRetrieval-Augmented\\nGeneration (RAG, Lewis et al., 2020b) enhances\\nlarge language models (LLMs) by 
retrieving rel-\\nevant document chunks from external knowledge\\nbases (Xu et al., 2025). RAG helps mitigate 
hal-\\nlucination by grounding generation in retrieved\\ncontent, and has become a foundational technique\\nin 
developing chatbots and real-world LLM ap-\\nplications (Gao et al., 2023b). Beyond the stan-\\ndard 
retrieve-then-generate pipeline (often referred\\nto as Naive RAG), numerous studies have pro-\\nposed techniques 
to improve performance and ef-\\nficiency. IRCoT (Trivedi et al\n[Quote from Retrieval-Augmented Generation for 
Knowledge-Intensive NLP Tasks] . Since RAG can be\\nemployed as a language model, similar concerns as for GPT-2 
[50] are valid here, although arguably\\nto a lesser extent, including that it might be used to generate abuse, 
faked or misleading content in\\nthe news or on social media; to impersonate others; or to automate the production 
of spam/phishing\\ncontent [54]. Advanced language models may also lead to the automation of various jobs in 
the\\ncoming decades [16]. In order to mitigate these risks, AI systems could be employed to \\ufb01ght 
against\\nmisleading content and automated spam/phishing.\\nAcknowledgments\\nThe authors would like to thank the 
reviewers for their thoughtful and constructive feedback on 

ChatPromptValue(
    messages=[
        SystemMessage(
            content='You are a document chatbot. Help the user as they ask questions about documents. User messaged
just asked: what is RECON\n\n From this, we have retrieved the following potentially-useful info:  Conversation 
History Retrieval:\n[Document(id=\'51577b9e-e303-48a8-9d23-4007ae7deba1\', metadata={}, page_content="Agent 
previously responded with RAG stands for Retrieval-Augmented Generation. It\'s a technique that enhances large 
language models by retrieving relevant document chunks from external knowledge bases. This helps to mitigate the 
issue of hallucination and has become a foundational technique in developing chatbots and real-world LLM 
applications. The retrieved content is grounded in the generation process, making the output more reliable and 
accurate. The original RAG model consists of two main steps: retrieval and generation. During the retrieval phase, 
a set of documents is gathered from a knowledge source based on the input prompt. Then, during the generation 
phase, the language model generates the output sequence using the retrieved documents as additional context. 
(Sources: Lewis et al., 2020b; Trivedi et al., 2023)"), Document(id=\'602a4fea-bd49-42b1-a334-6bbf6596c1d1\', 
metadata={}, page_content=\'Agent previously responded with RAG, or Retrieval-Augmented Generation, is a technique 
that enhances large language models by retrieving relevant document chunks from external knowledge bases. This 
helps to mitigate the issue of hallucination and has become a foundational technique in developing chatbots and 
real-world LLM applications. The retrieved content is grounded in the generation process, making the output more 
reliable and accurate.\\n\\nIn the original RAG model, there are two main steps: retrieval and generation. During 
the retrieval phase, a set of documents is gathered from a knowledge source based on the input prompt. Then, during
the generation phase, the language model generates the output sequence using the retrieved documents as additional 
context.\\n\\nThere are also variations of the RAG model like RAG-Sequence and RAG-Token. The RAG-Sequence model 
uses the same retrieved document to generate the complete sequence, while the RAG-Token model can predict each 
target token based on a different document.\\n\\nRecent improvements to RAG include IRCoT, which interleaves 
retrieval with steps in chain-of-thought prompting to enable more effective reasoning for knowledge-intensive tasks
that require multi-hop inference. Self-RAG introduces supervised fine-tuning to train the generator language model 
to autonomously decide whether to retrieve or generate at each step, achieving state-of-the-art performance on 
various QA and long-form generation benchmarks.\\n\\n(Sources: Lewis et al., 2020b; Trivedi et al.; Gao et al., 
2023a; Min et al., 2023; Chen et al., 2025; Song et al., 2025; Zheng et al., 2025; Jin et al., 2025c; Asai et al., 
2024)\'), Document(id=\'3138db96-6f3c-4cea-abc7-0fb0c60cee3a\', metadata={}, page_content=\'User previously 
responded with what is RAG\'), Document(id=\'901f7976-7223-4112-b597-66728d6be6a6\', metadata={}, 
page_content=\'User previously responded with Tell me about RAG!\')]\n\n Document Retrieval:\n[Quote from RECON: 
Reasoning with Condensation for Efficient Retrieval-Augmented Generation] . Further, RECON achieves\\nfaster 
training speed (5.2%) and lower inference la-\\ntency (30.9%) measured in wall-clock time, demon-\\nstrating that 
active context compression is critical\\ntoward practical, RL-augmented RAG systems.\\n2\\nRelated Works\\nRAG and 
Agentic RAG.\\nRetrieval-Augmented\\nGeneration (RAG, Lewis et al., 2020b) enhances\\nlarge language models (LLMs) 
by retrieving rel-\\nevant document chunks from external knowledge\\nbases (Xu et al., 2025). RAG helps mitigate 
hal-\\nlucination by grounding generation in retrieved\\ncontent, and has become a foundational technique\\nin 

ChatPromptValue(
    messages=[
        SystemMessage(
            content='You are a document chatbot. Help the user as they ask questions about documents. User messaged
just asked: what is IOT\n\n From this, we have retrieved the following potentially-useful info:  Conversation 
History Retrieval:\n[Document(id=\'602a4fea-bd49-42b1-a334-6bbf6596c1d1\', metadata={}, page_content=\'Agent 
previously responded with RAG, or Retrieval-Augmented Generation, is a technique that enhances large language 
models by retrieving relevant document chunks from external knowledge bases. This helps to mitigate the issue of 
hallucination and has become a foundational technique in developing chatbots and real-world LLM applications. The 
retrieved content is grounded in the generation process, making the output more reliable and accurate.\\n\\nIn the 
original RAG model, there are two main steps: retrieval and generation. During the retrieval phase, a set of 
documents is gathered from a knowledge source based on the input prompt. Then, during the generation phase, the 
language model generates the output sequence using the retrieved documents as additional context.\\n\\nThere are 
also variations of the RAG model like RAG-Sequence and RAG-Token. The RAG-Sequence model uses the same retrieved 
document to generate the complete sequence, while the RAG-Token model can predict each target token based on a 
different document.\\n\\nRecent improvements to RAG include IRCoT, which interleaves retrieval with steps in 
chain-of-thought prompting to enable more effective reasoning for knowledge-intensive tasks that require multi-hop 
inference. Self-RAG introduces supervised fine-tuning to train the generator language model to autonomously decide 
whether to retrieve or generate at each step, achieving state-of-the-art performance on various QA and long-form 
generation benchmarks.\\n\\n(Sources: Lewis et al., 2020b; Trivedi et al.; Gao et al., 2023a; Min et al., 2023; 
Chen et al., 2025; Song et al., 2025; Zheng et al., 2025; Jin et al., 2025c; Asai et al., 2024)\'), 
Document(id=\'51577b9e-e303-48a8-9d23-4007ae7deba1\', metadata={}, page_content="Agent previously responded with 
RAG stands for Retrieval-Augmented Generation. It\'s a technique that enhances large language models by retrieving 
relevant document chunks from external knowledge bases. This helps to mitigate the issue of hallucination and has 
become a foundational technique in developing chatbots and real-world LLM applications. The retrieved content is 
grounded in the generation process, making the output more reliable and accurate. The original RAG model consists 
of two main steps: retrieval and generation. During the retrieval phase, a set of documents is gathered from a 
knowledge source based on the input prompt. Then, during the generation phase, the language model generates the 
output sequence using the retrieved documents as additional context. (Sources: Lewis et al., 2020b; Trivedi et al.,
2023)"), Document(id=\'ca8c054c-fa48-4068-9ac2-8529a483391e\', metadata={}, page_content=\'Agent previously 
responded with RECON stands for "Reasoning with Condensation for Efficient Retrieval-Augmented Generation." It\\\'s
a framework designed to improve the efficiency of retrieval-augmented generation systems, particularly those that 
use reinforcement learning (RL) for training, by addressing context management challenges.\\n\\nWhen RL-based 
retrieval-augmented generation systems are trained with long and noisy retrieved documents, it can lead to 
increased costs and degraded performance due to inefficient context management. RECON aims to tackle this issue by 
incorporating an explicit summarization module within the reasoning loop to compress evidence. This distillation 
stage transfers multi-aspect supervision into a smaller, efficient model suitable for integration into RL-based 
RAG.\\n\\nThey specifically mention the integration of RECON with the Search-R1 pipeline, where retrie

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://80a97684fea93d9699.gradio.live
Closing server running on port: 7860


<br>

----

<br>

## **Part 4:** Saving Your Index For Evaluation

After you've implemented your RAG chain, please save your accumulated vector store as shown [in the official documentation](https://python.langchain.com/docs/integrations/vectorstores/faiss#saving-and-loading). You'll have a chance to use it again for your final assessment!

In [18]:
## Save and compress your index
docstore.save_local("docstore_index")
!tar czvf docstore_index.tgz docstore_index

!rm -rf docstore_index

docstore_index/
docstore_index/index.pkl
docstore_index/index.faiss


If everything was properly saved, the following line can be invoked to pull the index from the compressed `tgz` file (assuming the pip requirements are installed). After you have confirmed that the cell can pull in your index, download `docstore_index.tgz` for use in the last notebook!

In [19]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.vectorstores import FAISS

# embedder = NVIDIAEmbeddings(model="nvidia/nv-embed-v1", truncate="END")
!tar xzvf docstore_index.tgz
new_db = FAISS.load_local("docstore_index", embedder, allow_dangerous_deserialization=True)
docs = new_db.similarity_search("Testing the index")
print(docs[0].page_content[:1000])

docstore_index/
docstore_index/index.pkl
docstore_index/index.faiss
. To demonstrate, we build an index using the DrQA [5]\nWikipedia dump from December 2016 and compare outputs from RAG using this index to the newer\nindex from our main results (December 2018). We prepare a list of 82 world leaders who had changed\n7\nTable 4: Human assessments for the Jeopardy\nQuestion Generation Task.\nFactuality\nSpeci\ufb01city\nBART better\n7.1%\n16.8%\nRAG better\n42.7%\n37.4%\nBoth good\n11.7%\n11.8%\nBoth poor\n17.7%\n6.9%\nNo majority\n20.8%\n20.1%\nTable 5: Ratio of distinct to total tri-grams for\ngeneration tasks.\nMSMARCO\nJeopardy QGen\nGold\n89.6%\n90.0%\nBART\n70.7%\n32.4%\nRAG-Token\n77.8%\n46.8%\nRAG-Seq.\n83.5%\n53.8%\nTable 6: Ablations on the dev set. As FEVER is a classi\ufb01cation task, both RAG models are equivalent.\nModel\nNQ\nTQA\nWQ\nCT\nJeopardy-QGen\nMSMarco\nFVR-3\nFVR-2\nExact Match\nB-1\nQB-1\nR-L\nB-1\nLabel Accuracy\nRAG-Token-BM25\n29.7\n41.5\n32.1\n33.1\n17.5\n22

-----

<br>

## **Part 5:** Wrap-Up

Congratulations! Assuming your RAG chain is all good, you're now ready to move on to the **RAG Evaluation [Assessment]** section!

### <font color="#76b900">**Great Job!**</font>

### **Next Steps:**
1. **[Optional]** Revisit the **"Questions To Think About" Section** at the top of the notebook and think about some possible answers.

---

<center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/DLI_Header_White.png" width="400" height="186" /></a></center>